# Compare All 4 Dual Next Activity Predictors (BPIC17)

This notebook compares the four dual lifecycle next-activity predictors for use in the simulation engine:

1. **start_complete/baseline** – trained on start+complete events only
2. **start_complete/balanced** – start+complete with class-balanced weighting
3. **full_lifecycle/baseline** – trained on all lifecycle transitions
4. **full_lifecycle/balanced** – full lifecycle with class-balanced weighting

For each predictor it runs simulation against BPIC17, then benchmarks all four using `SimulationBenchmark` and produces a side-by-side comparison.

## Usage

- **Run from repo root** so that `integration`, `Dataset`, and `next_activity_prediction_lifecycle_dual` are available.
- **Prerequisites:** BPIC17 XES at `Dataset/BPI Challenge 2017.xes`; dual models trained under `next_activity_prediction_lifecycle_dual/models/`; pm4py, pandas, resources.
- **Config (config cell):** Set `RUN_SIMULATIONS=True` to run simulations for all 4 predictors; `False` to benchmark only pre-saved logs in `integration/output/dual_predictor_comparison/`. Adjust `NUM_CASES` and `FILTER_LIFECYCLE_COMPLETE` as needed.
- **Baseline models:** If not trained locally, the integration will auto-download from HuggingFace (`huggingface_hub` required). **Balanced** models must be trained locally.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "integration").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from integration.SimulationBenchmark import SimulationBenchmark
from integration.config import SimulationConfig
from integration.setup import setup_simulation
from simulation.engine import DESEngine
from simulation.log_exporter import LogExporter

print("Repo root:", repo_root)

Repo root: D:\Repos\process-simulation-engine-1


In [2]:
ORIGINAL_LOG_PATH = repo_root / "Dataset" / "BPI Challenge 2017.xes"
OUTPUT_DIR = repo_root / "integration" / "output" / "dual_predictor_comparison"
NUM_CASES = 100
RUN_SIMULATIONS = True  # Set False to use pre-saved simulated logs only
FILTER_LIFECYCLE_COMPLETE = False  # True = compare start+complete only

DUAL_PREDICTORS = [
    ("start_complete_baseline", "start_complete/baseline"),
    ("start_complete_balanced", "start_complete/balanced"),
    ("full_lifecycle_baseline", "full_lifecycle/baseline"),
    ("full_lifecycle_balanced", "full_lifecycle/balanced"),
]

MODEL_BASE = repo_root / "next_activity_prediction_lifecycle_dual" / "models"

if not ORIGINAL_LOG_PATH.exists():
    raise FileNotFoundError(f"Original log not found: {ORIGINAL_LOG_PATH}")

In [3]:
def run_simulation_for_predictor(config: SimulationConfig, df, allocator, output_path: Path):
    from datetime import datetime
    import os

    start_date = pd.to_datetime(df["time:timestamp"]).min().to_pydatetime()
    arrivals, next_act_pred, proc_pred, attr_pred = setup_simulation(
        config, df=df, start_date=start_date
    )
    engine_start_time = min(arrivals[0], start_date) if arrivals else start_date

    engine = DESEngine(
        resource_allocator=allocator,
        arrival_timestamps=arrivals,
        next_activity_predictor=next_act_pred,
        processing_time_predictor=proc_pred,
        case_attribute_predictor=attr_pred,
        start_time=engine_start_time,
    )
    events = engine.run(num_cases=len(arrivals))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    LogExporter.to_csv(events, str(output_path))
    return events

if RUN_SIMULATIONS:
    from integration.test_integration import load_event_log, create_resource_allocator

    df = load_event_log(str(ORIGINAL_LOG_PATH))
    allocator = create_resource_allocator(str(ORIGINAL_LOG_PATH))
    base_config = SimulationConfig.all_advanced(
        event_log_path=str(ORIGINAL_LOG_PATH), num_cases=NUM_CASES
    )
    base_config.num_cases = NUM_CASES
    base_config.next_activity_class = "lstm"
    base_config.next_activity_mode = "advanced"
    base_config.next_activity_model_type = "lifecycle_dual"

    for key, subpath in DUAL_PREDICTORS:
        model_path = MODEL_BASE / subpath
        has_local = (model_path / "model.keras").exists() or (model_path / "checkpoints" / "best_model.keras").exists()
        if "balanced" in subpath and not has_local:
            print(f"Skipping {key}: balanced models require local training (no HuggingFace)")
            continue
        base_config.next_activity_model_path = str(model_path)
        out_path = OUTPUT_DIR / key / "simulated_log.csv"
        print(f"Running simulation: {key}...")
        try:
            run_simulation_for_predictor(base_config, df, allocator, out_path)
            print(f"  -> {out_path}")
        except Exception as e:
            print(f"  Skipped {key}: {e}")
else:
    print("RUN_SIMULATIONS=False: using pre-saved logs from", OUTPUT_DIR)

Loaded event log: 1202267 events, 31509 cases
[AUTO-LOAD] Found cached model at D:\Repos\process-simulation-engine-1\resources\resource_availabilities\bpic2017_resource_model.pkl
            Loading pre-trained model (fast)...
[LOADING] Loading model from: D:\Repos\process-simulation-engine-1\resources\resource_availabilities\bpic2017_resource_model.pkl
[SUCCESS] Model loaded
          - 144 resource patterns
          - 141 clustered resources
          - 0 busy periods
[AUTO-LOAD] Model loaded successfully!
Loaded ResourceAllocator from event log
Skipping start_complete_baseline: model not found at D:\Repos\process-simulation-engine-1\next_activity_prediction_lifecycle_dual\models\start_complete\baseline
Skipping start_complete_balanced: model not found at D:\Repos\process-simulation-engine-1\next_activity_prediction_lifecycle_dual\models\start_complete\balanced
Skipping full_lifecycle_baseline: model not found at D:\Repos\process-simulation-engine-1\next_activity_prediction_lifecycl

In [4]:
results_by_predictor = {}

for key, _ in DUAL_PREDICTORS:
    sim_path = OUTPUT_DIR / key / "simulated_log.csv"
    if not sim_path.exists():
        print(f"Skipping {key}: {sim_path} not found")
        continue
    benchmark = SimulationBenchmark(
        original_log=str(ORIGINAL_LOG_PATH),
        simulated_log=str(sim_path),
        filter_lifecycle_complete=FILTER_LIFECYCLE_COMPLETE,
    )
    results_by_predictor[key] = benchmark.compute_all_metrics()
    print(f"Benchmarked {key}")

Skipping start_complete_baseline: D:\Repos\process-simulation-engine-1\integration\output\dual_predictor_comparison\start_complete_baseline\simulated_log.csv not found
Skipping start_complete_balanced: D:\Repos\process-simulation-engine-1\integration\output\dual_predictor_comparison\start_complete_balanced\simulated_log.csv not found
Skipping full_lifecycle_baseline: D:\Repos\process-simulation-engine-1\integration\output\dual_predictor_comparison\full_lifecycle_baseline\simulated_log.csv not found
Skipping full_lifecycle_balanced: D:\Repos\process-simulation-engine-1\integration\output\dual_predictor_comparison\full_lifecycle_balanced\simulated_log.csv not found


In [5]:
metrics_df = pd.DataFrame(
    {
        key: r["simple_metrics"].set_index("Metric")["Percentage"]
        for key, r in results_by_predictor.items()
    }
)
metrics_df.index.name = "Metric"
metrics_df

""
Metric


In [6]:
summary_rows = []
for key, r in results_by_predictor.items():
    bs = r["basic_stats"]
    row = {"Predictor": key}
    for _, rw in bs.iterrows():
        log_type = rw["Log"].lower()
        if log_type == "simulated":
            row["Events"] = rw["Number of Events"]
            row["Cases"] = rw["Number of Cases"]
            row["Activities"] = rw["Number of Unique Activities"]
    summary_rows.append(row)
pd.DataFrame(summary_rows)

""


In [7]:
if len(results_by_predictor) >= 2:
    key_metrics = [
        "Activity Jaccard Similarity",
        "Trace Variant Jaccard Similarity",
        "DFG Edge Jaccard Similarity",
        "Top 10 Variant Overlap",
        "Top 20 DFG Edge Overlap",
    ]
    plot_df = metrics_df.loc[[m for m in key_metrics if m in metrics_df.index]]
    if not plot_df.empty:
        ax = plot_df.T.plot(kind="bar", figsize=(10, 5), width=0.8)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right")
        ax.set_ylabel("Percentage")
        ax.set_title("Dual predictor comparison vs BPIC17")
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.show()